# LSTM-only baseline — XSS & SQL Injection

Đổi runtime sang **T4 GPU**, rồi chạy từng cell.

In [1]:
%pip -q install lightning openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 25.4 MB/s eta 0:00:00


In [4]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = Path("/content/drive/MyDrive")
required_files = {
    "obfuscation_dataset_full.xlsx",
    "SQLInjection_XSS_MixDataset.1.0.0.xlsx",
    "csic_database.csv",
}

matches = [
    folder
    for folder in DRIVE_ROOT.rglob("*")
    if folder.is_dir() and required_files.issubset({item.name for item in folder.iterdir()})
]

if not matches:
    raise FileNotFoundError(
        f"Không tìm thấy folder có đủ 3 dataset trong {DRIVE_ROOT}. "
        "Kiểm tra lại file đã nằm trong My Drive chưa."
    )

DRIVE_DATA_DIR = matches[0]
print("Đã tìm thấy dataset tại:", DRIVE_DATA_DIR)
print([file.name for file in DRIVE_DATA_DIR.iterdir()])

os.environ["LSTM_DATA_DIR"] = str(DRIVE_DATA_DIR)
os.environ["LSTM_OUTPUT_DIR"] = "/content/drive/MyDrive/LSTM-Only/outputs/lstm_baseline"

Mounted at /content/drive
Đã tìm thấy dataset tại: /content/drive/MyDrive/LSTM-Only/data
['csic_database.csv', 'obfuscation_dataset_full.xlsx', 'SQLInjection_XSS_MixDataset.1.0.0.xlsx']


## Pipeline

Copy toàn bộ nội dung của `lstm_only_baseline.py` vào cell code ngay bên dưới và chạy. Script dùng folder Drive đã cấu hình ở cell trước và lưu output bền vững trong `MyDrive/NCKH-LSTM/outputs/lstm_baseline/`.

In [5]:
"""Standalone LSTM-only baseline for obfuscated XSS / SQL injection detection.

Install (in a working virtual environment):
    pip install pandas numpy scikit-learn torch lightning matplotlib openpyxl

Run from this directory or the repository root:
    python lstm_only_baseline.py

Markdown note
-------------
Mô hình LSTM-only được xây dựng như một baseline độc lập nhằm đánh giá khả
năng học quan hệ tuần tự trong chuỗi payload. Pipeline sử dụng character-level
encoding vì payload XSS và SQL Injection bị che giấu thường thay đổi ở cấp ký
tự. Khác với preprocessing NLP thông thường, pipeline giữ lại ký tự đặc biệt
vì đây là tín hiệu quan trọng cho phát hiện tấn công.
"""

# ============================================================
# 1. Imports
# ============================================================
from __future__ import annotations

import html
import json
import os
import random
import re
import sys
import unicodedata
from pathlib import Path
from typing import Iterable
from urllib.parse import unquote

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import Callback, EarlyStopping, ModelCheckpoint
    from lightning.pytorch.loggers import CSVLogger
except ImportError as exc:
    raise ImportError("Install dependencies with: pip install torch lightning") from exc


# ============================================================
# 2. Config
# ============================================================
# Works both as a local script and when the whole pipeline is pasted into a
# Google Colab notebook cell (where __file__ is not defined).
IS_COLAB = "google.colab" in sys.modules
ROOT = Path("/content") if IS_COLAB else Path(__file__).resolve().parent
DATA_DIR = Path(os.environ.get("LSTM_DATA_DIR", str(ROOT)))
DATA_FILES = [
    DATA_DIR / "obfuscation_dataset_full.xlsx",
    DATA_DIR / "SQLInjection_XSS_MixDataset.1.0.0.xlsx",
    DATA_DIR / "csic_database.csv",
]
OUTPUT_DIR = Path(os.environ.get("LSTM_OUTPUT_DIR", str(ROOT / "outputs" / "lstm_baseline")))
SEED = 42
MAX_LEN = 400
BATCH_SIZE = 128
NUM_WORKERS = 0
EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
USE_RAW_PLUS_NORMALIZED = True
LOWERCASE = True

# Set a mapping here only when automatic column detection raises an ambiguity error.
# Example: COLUMN_OVERRIDES = {"obfuscation_dataset_full.xlsx": ("payload", "label")}
COLUMN_OVERRIDES: dict[str, tuple[str, str]] = {}
TEXT_HINTS = ("payload", "request", "url", "uri", "query", "text", "content", "body", "data", "sentence", "input")
LABEL_HINTS = ("label", "class", "category", "type", "target", "attack", "malicious")
HTTP_HINTS = ("method", "url", "uri", "path", "query", "body", "content", "request", "parameter", "header")
BENIGN_LABELS = {"normal", "benign", "clean", "safe", "legitimate", "valid", "0", "false", "no"}
ATTACK_LABELS = {"attack", "anomaly", "anomalous", "malicious", "xss", "sqli", "sql injection", "injection", "invalid", "1", "true", "yes"}


# ============================================================
# 3. Reproducibility and Load/Standardize Datasets
# ============================================================
def seed_everything(seed: int) -> None:
    pl.seed_everything(seed, workers=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, engine="openpyxl")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported dataset type: {path.suffix}")


def _matches(columns: Iterable[str], hints: Iterable[str]) -> list[str]:
    return [col for col in columns if any(h in col.lower() for h in hints)]


def _one_hot_label_columns(columns: Iterable[str]) -> tuple[list[str], list[str]]:
    """Return candidate attack and benign columns for one-hot-labelled datasets."""
    attack, benign = [], []
    for column in columns:
        normalized = re.sub(r"[_\-]+", " ", column.lower()).strip()
        if any(token in normalized for token in ("xss", "sql", "inject", "attack", "malicious", "anomal")):
            attack.append(column)
        elif any(token in normalized for token in ("normal", "benign", "clean", "safe", "legitimate", "valid")):
            benign.append(column)
    return attack, benign


def choose_columns(df: pd.DataFrame, path: Path) -> tuple[str, str]:
    if path.name in COLUMN_OVERRIDES:
        return COLUMN_OVERRIDES[path.name]
    cols = [str(c) for c in df.columns]
    # CSIC has an explicit numeric classification field plus fragmented HTTP
    # fields.  Check this first: names such as "content-type" would otherwise
    # be accidental matches for generic text/type heuristics.
    if path.name.lower().startswith("csic"):
        classification = next((col for col in cols if col.lower().strip() == "classification"), None)
        http_cols = _matches(cols, HTTP_HINTS)
        if classification and http_cols:
            return "__CONCAT_HTTP_FIELDS__", classification
    text_candidates, label_candidates = _matches(cols, TEXT_HINTS), _matches(cols, LABEL_HINTS)
    # An exact "label" is intentionally preferred over incidental names such as
    # "classification" when both occur in a source table.
    exact_label = next((col for col in cols if col.lower().strip() == "label"), None)
    if len(text_candidates) == 1 and exact_label:
        return text_candidates[0], exact_label
    if len(text_candidates) == 1 and len(label_candidates) == 1:
        return text_candidates[0], label_candidates[0]
    attack_cols, benign_cols = _one_hot_label_columns(cols)
    if len(text_candidates) == 1 and attack_cols and benign_cols:
        return text_candidates[0], "__ONE_HOT_LABELS__"
    raise ValueError(
        f"Could not unambiguously identify text/label columns in {path.name}.\n"
        f"Columns: {cols}\nText candidates: {text_candidates}\nLabel candidates: {label_candidates}\n"
        "Set COLUMN_OVERRIDES[filename] = (TEXT_COLUMN, LABEL_COLUMN) in Config."
    )


def normalize_label(value: object) -> int:
    value_s = str(value).strip().lower()
    value_s = re.sub(r"\s+", " ", value_s)
    if value_s in BENIGN_LABELS or value_s in {"0.0", "0.00"}:
        return 0
    if value_s in ATTACK_LABELS or value_s in {"1.0", "1.00"} or any(token in value_s for token in ("xss", "sql", "inject", "attack", "malicious", "anomal")):
        return 1
    raise ValueError(f"Unrecognized label {value!r}. Extend BENIGN_LABELS/ATTACK_LABELS explicitly.")


def standardize_dataset(path: Path) -> pd.DataFrame:
    print(f"Loading {path.name} ...", flush=True)
    raw = read_table(path)
    text_col, label_col = choose_columns(raw, path)
    if text_col == "__CONCAT_HTTP_FIELDS__":
        http_cols = _matches([str(c) for c in raw.columns], HTTP_HINTS)
        text = raw[http_cols].fillna("").astype(str).agg(" ".join, axis=1)
    else:
        text = raw[text_col]
    if label_col == "__ONE_HOT_LABELS__":
        attack_cols, benign_cols = _one_hot_label_columns([str(c) for c in raw.columns])
        # Any active XSS/SQLi/attack indicator is Attack; otherwise active Normal is benign.
        attack_active = raw[attack_cols].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).any(axis=1)
        benign_active = raw[benign_cols].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).any(axis=1)
        invalid = ~(attack_active | benign_active)
        if invalid.any():
            raise ValueError(f"{path.name}: {invalid.sum()} rows have no active one-hot class in {attack_cols + benign_cols}.")
        original_label = np.where(attack_active, "Attack", "Normal")
        labels = attack_active.astype(int)
    else:
        original_label = raw[label_col]
        labels = None
    result = pd.DataFrame({"text": text, "original_label": original_label, "source_dataset": path.name})
    result = result.dropna(subset=["text", "original_label"])
    result["text"] = result["text"].astype(str).str.strip()
    result = result[result["text"].ne("")].copy()
    try:
        result["label"] = labels.loc[result.index].to_numpy() if labels is not None else result["original_label"].map(normalize_label)
    except ValueError as exc:
        unique = result["original_label"].drop_duplicates().head(30).tolist()
        raise ValueError(f"{path.name}: {exc}\nExample original labels: {unique}") from exc
    result = result[["text", "label", "original_label", "source_dataset"]]
    print(f"  kept {len(result):,} labelled rows", flush=True)
    return result


def load_all_datasets(paths: list[Path]) -> pd.DataFrame:
    merged = pd.concat([standardize_dataset(p) for p in paths], ignore_index=True)
    merged = merged.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    print(f"Total unique samples: {len(merged):,}")
    print("Overall label distribution:\n", merged["label"].value_counts().sort_index())
    print("Label distribution by source:\n", pd.crosstab(merged["source_dataset"], merged["label"]))
    return merged


# ============================================================
# 4. Payload Preprocessing and Character Encoding
# ============================================================
def normalize_payload_for_lstm(text: object) -> str:
    """Decode common obfuscation while preserving security-significant punctuation."""
    value = unicodedata.normalize("NFKC", str(text)).strip()
    value = html.unescape(value)
    for _ in range(2):
        decoded = unquote(value)
        if decoded == value:
            break
        value = decoded
    if LOWERCASE:
        value = value.lower()
    value = "".join(ch for ch in value if not unicodedata.category(ch).startswith("C") or ch in "\n\t")
    return re.sub(r"\s+", " ", value).strip()


def prepare_model_text(raw: object) -> str:
    raw_text = str(raw).strip()
    normalized = normalize_payload_for_lstm(raw_text)
    return f"{raw_text} [SEP] {normalized}" if USE_RAW_PLUS_NORMALIZED else normalized


def sequence_stats(texts: pd.Series) -> None:
    lengths = texts.str.len()
    print("Character lengths:", {"min": int(lengths.min()), "mean": round(float(lengths.mean()), 2),
          "median": float(lengths.median()), "p95": float(lengths.quantile(.95)), "max": int(lengths.max())})


def build_vocab(texts: Iterable[str]) -> dict[str, int]:
    chars = sorted(set("".join(texts)))
    return {"<PAD>": 0, "<UNK>": 1, **{char: i + 2 for i, char in enumerate(chars)}}


def encode_text(text: str, char2idx: dict[str, int]) -> np.ndarray:
    ids = [char2idx.get(char, 1) for char in text[:MAX_LEN]]
    return np.pad(np.asarray(ids, dtype=np.int64), (0, MAX_LEN - len(ids)), constant_values=0)


# ============================================================
# 5. Split Data, Dataset, and DataLoaders
# ============================================================
class PayloadDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, char2idx: dict[str, int]):
        self.features = np.stack([encode_text(text, char2idx) for text in frame["text_for_model"]])
        self.labels = frame["label"].to_numpy(dtype=np.float32)
        # This lets the LSTM stop at the final real character rather than
        # consuming a long suffix of PAD tokens.
        self.lengths = frame["text_for_model"].map(lambda text: max(1, min(len(text), MAX_LEN))).to_numpy(dtype=np.int64)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return torch.from_numpy(self.features[idx]), torch.tensor(self.labels[idx]), torch.tensor(self.lengths[idx])


def split_data(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Warning: obfuscated variants of one base payload should be grouped by base_payload_id,
    # if available, otherwise random splitting may leak closely related samples across splits.
    train, holdout = train_test_split(frame, test_size=.30, stratify=frame["label"], random_state=SEED)
    val, test = train_test_split(holdout, test_size=.50, stratify=holdout["label"], random_state=SEED)
    for name, part in (("train", train), ("validation", val), ("test", test)):
        print(f"{name} ({len(part):,}) label distribution:\n{part['label'].value_counts().sort_index()}")
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def make_loader(frame: pd.DataFrame, vocab: dict[str, int], shuffle: bool) -> DataLoader:
    return DataLoader(PayloadDataset(frame, vocab), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


# ============================================================
# 6. LSTM-only Model (no CNN, attention, Transformer, or BiLSTM)
# ============================================================
class LSTMOnlyClassifier(pl.LightningModule):
    def __init__(self, vocab_size: int, pos_weight: float):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=0)
        self.embedding_dropout = nn.Dropout(.2)
        self.lstm = nn.LSTM(EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, batch_first=True, dropout=.3, bidirectional=False)
        self.classifier = nn.Sequential(nn.Linear(HIDDEN_SIZE, 64), nn.ReLU(), nn.Dropout(.3), nn.Linear(64, 1))
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

    def forward(self, token_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding_dropout(self.embedding(token_ids))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        return self.classifier(hidden[-1]).squeeze(1)

    def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        ids, targets, lengths = batch
        logits = self(ids, lengths)
        loss = self.loss_fn(logits, targets)
        probs, pred = torch.sigmoid(logits), (torch.sigmoid(logits) >= .5).int()
        self.log(f"{stage}_loss", loss, on_epoch=True, prog_bar=(stage == "val"), batch_size=len(targets))
        if stage == "val":
            self.log("val_accuracy", (pred == targets.int()).float().mean(), on_epoch=True, batch_size=len(targets))
        return loss, probs, targets

    def training_step(self, batch, batch_idx):
        loss, _, _ = self._step(batch, "train")
        return loss

    def validation_step(self, batch, batch_idx):
        loss, probs, targets = self._step(batch, "val")
        return {"loss": loss, "probs": probs.detach(), "targets": targets.detach()}

    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


class ValidationMetrics(Callback):
    """Computes epoch-level threshold-0.5 validation metrics from all batches."""
    def on_validation_epoch_start(self, trainer, pl_module): self.probs, self.targets = [], []
    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        self.probs.extend(outputs["probs"].cpu().tolist())
        self.targets.extend(outputs["targets"].cpu().int().tolist())
    def on_validation_epoch_end(self, trainer, pl_module):
        if not self.targets or len(set(self.targets)) < 2: return
        y, p = np.array(self.targets), np.array(self.probs); pred = (p >= .5).astype(int)
        metrics = {"val_precision": precision_score(y, pred, zero_division=0), "val_recall": recall_score(y, pred, zero_division=0),
                   "val_f1": f1_score(y, pred, zero_division=0), "val_roc_auc": roc_auc_score(y, p), "val_pr_auc": average_precision_score(y, p)}
        for name, value in metrics.items(): pl_module.log(name, value, on_epoch=True, prog_bar=(name == "val_pr_auc"))


class History(Callback):
    def __init__(self): self.rows: list[dict[str, float]] = []
    def on_validation_epoch_end(self, trainer, pl_module):
        row = {"epoch": trainer.current_epoch}
        for key, value in trainer.callback_metrics.items():
            if key in {"train_loss", "val_loss", "val_pr_auc", "val_f1"}: row[key] = float(value.detach().cpu())
        self.rows.append(row)


# ============================================================
# 7. Train and Save Curves
# ============================================================
def plot_training_curves(history: list[dict[str, float]], path: Path) -> None:
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key in ("train_loss", "val_loss"):
        if key in frame: axes[0].plot(frame["epoch"], frame[key], label=key)
    for key in ("val_pr_auc", "val_f1"):
        if key in frame: axes[1].plot(frame["epoch"], frame[key], label=key)
    axes[0].set_title("Loss"); axes[1].set_title("Validation metrics")
    for ax in axes: ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


# ============================================================
# 8. Evaluation, Threshold Tuning, and Artifacts
# ============================================================
@torch.inference_mode()
def predict(model: LSTMOnlyClassifier, loader: DataLoader) -> np.ndarray:
    model.eval(); values = []
    device = model.device
    for ids, _, lengths in loader:
        values.extend(torch.sigmoid(model(ids.to(device), lengths)).cpu().numpy())
    return np.asarray(values)


def metrics_at_threshold(y_true: np.ndarray, probs: np.ndarray, threshold: float) -> dict:
    pred = (probs >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {"threshold": float(threshold), "accuracy": float(accuracy_score(y_true, pred)),
            "precision": float(precision_score(y_true, pred, zero_division=0)), "recall": float(recall_score(y_true, pred, zero_division=0)),
            "f1": float(f1_score(y_true, pred, zero_division=0)), "roc_auc": float(roc_auc_score(y_true, probs)),
            "pr_auc": float(average_precision_score(y_true, probs)), "false_positive": int(cm[0, 1]),
            "false_negative": int(cm[1, 0]), "attack_recall": float(recall_score(y_true, pred, pos_label=1, zero_division=0)),
            "attack_f1": float(f1_score(y_true, pred, pos_label=1, zero_division=0)), "confusion_matrix": cm.tolist()}


def evaluate_model(y_true: np.ndarray, probs: np.ndarray, threshold: float, title: str) -> tuple[dict, dict]:
    """Evaluate the held-out test set; Attack Recall/F1 matter most for missed attacks."""
    results = metrics_at_threshold(y_true, probs, threshold)
    report = classification_report(
        y_true, probs >= threshold, target_names=["Normal", "Attack"], zero_division=0, output_dict=True
    )
    print(f"\n{title}")
    for name in ("accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "false_positive", "false_negative", "attack_recall", "attack_f1"):
        print(f"{name}: {results[name]}")
    print("Confusion matrix:\n", np.asarray(results["confusion_matrix"]))
    print("Classification report:\n", classification_report(y_true, probs >= threshold, target_names=["Normal", "Attack"], zero_division=0))
    return results, report


def best_attack_threshold(y: np.ndarray, probs: np.ndarray) -> float:
    candidates = np.arange(.05, .951, .01)
    return float(max(candidates, key=lambda t: f1_score(y, probs >= t, zero_division=0)))


def save_confusion_matrix(cm: list[list[int]], path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5, 4)); image = ax.imshow(cm, cmap="Blues")
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], xlabel="Predicted", ylabel="Actual")
    for i in range(2):
        for j in range(2): ax.text(j, i, cm[i][j], ha="center", va="center")
    fig.colorbar(image); fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


# ============================================================
# 9. Main
# ============================================================
def main() -> None:
    seed_everything(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df = load_all_datasets(DATA_FILES)
    df["text_for_model"] = df["text"].map(prepare_model_text)
    train, val, test = split_data(df)
    sequence_stats(train["text_for_model"])
    char2idx = build_vocab(train["text_for_model"])
    print(f"Vocabulary size (train only): {len(char2idx):,}")
    with (OUTPUT_DIR / "char2idx.json").open("w", encoding="utf-8") as handle: json.dump(char2idx, handle, ensure_ascii=False, indent=2)
    train_loader = make_loader(train, char2idx, shuffle=True)
    val_loader = make_loader(val, char2idx, shuffle=False)
    test_loader = make_loader(test, char2idx, shuffle=False)
    positives, negatives = int(train.label.sum()), int((train.label == 0).sum())
    pos_weight = negatives / max(positives, 1)
    history, metrics_callback = History(), ValidationMetrics()
    checkpoint = ModelCheckpoint(dirpath=OUTPUT_DIR, filename="best_lstm_model", monitor="val_pr_auc", mode="max", save_top_k=1)
    trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="auto", devices=1, deterministic=True, logger=CSVLogger(OUTPUT_DIR, name="lightning_logs"),
                         callbacks=[metrics_callback, history, EarlyStopping(monitor="val_pr_auc", mode="max", patience=6), checkpoint], log_every_n_steps=10)
    model = LSTMOnlyClassifier(len(char2idx), pos_weight)
    trainer.fit(model, train_loader, val_loader)
    best_model = LSTMOnlyClassifier.load_from_checkpoint(checkpoint.best_model_path)
    plot_training_curves(history.rows, OUTPUT_DIR / "training_curves.png")
    # Tune only on validation data; test remains an unbiased final comparison set.
    val_probs, y_val = predict(best_model, val_loader), val.label.to_numpy()
    tuned = best_attack_threshold(y_val, val_probs)
    test_probs, y_test = predict(best_model, test_loader), test.label.to_numpy()
    threshold_05, report_05 = evaluate_model(y_test, test_probs, .5, "Test metrics at threshold 0.5")
    best, report_best = evaluate_model(y_test, test_probs, tuned, f"Test metrics at validation-tuned threshold {tuned:.2f}")
    # In security detection, false negatives are dangerous missed attacks; prioritize Attack Recall and Attack F1 over accuracy.
    save_confusion_matrix(best["confusion_matrix"], OUTPUT_DIR / "confusion_matrix.png")
    pd.DataFrame({"text": test.text, "label": y_test, "original_label": test.original_label, "source_dataset": test.source_dataset,
                  "prob_attack": test_probs, "pred_at_0_5": (test_probs >= .5).astype(int),
                  "pred_at_best_threshold": (test_probs >= tuned).astype(int)}).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
    config = {k: v for k, v in globals().items() if k in {"SEED", "MAX_LEN", "BATCH_SIZE", "EMBEDDING_DIM", "HIDDEN_SIZE", "NUM_LAYERS", "LEARNING_RATE", "USE_RAW_PLUS_NORMALIZED", "LOWERCASE"}}
    with (OUTPUT_DIR / "config.json").open("w") as handle: json.dump(config, handle, indent=2)
    evaluation = {"threshold_0_5": {"metrics": threshold_05, "classification_report": report_05},
                  "best_attack_threshold": {"selected_on": "validation", "threshold": tuned, "metrics": best, "classification_report": report_best}}
    with (OUTPUT_DIR / "evaluation_results.json").open("w") as handle: json.dump(evaluation, handle, indent=2)


if __name__ == "__main__":
    main()


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Loading obfuscation_dataset_full.xlsx ...
  kept 150,000 labelled rows
Loading SQLInjection_XSS_MixDataset.1.0.0.xlsx ...
  kept 156,016 labelled rows
Loading csic_database.csv ...
  kept 61,065 labelled rows
Total unique samples: 327,002
Overall label distribution:
 label
0     63321
1    263681
Name: count, dtype: int64
Label distribution by source:
 label                                       0       1
source_dataset                                       
SQLInjection_XSS_MixDataset.1.0.0.xlsx  53677   97717
csic_database.csv                        9644   15964
obfuscation_dataset_full.xlsx               0  150000
train (228,901) label distribution:
label
0     44325
1    184576
Name: count, dtype: int64
validation (49,050) label distribution:
label
0     9498
1    39552
Name: count, dtype: int64
test (49,051) label distribution:
label
0     9498
1    39553
Name: count, dtype: int64
Character lengths: {'min': 9, 'mean': 461.6, 'median': 219.0, 'p95': 1627.0, 'max': 13631}
Vocabulary

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/LSTM-Only/outputs/lstm_baseline exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVI

┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding         │ Embedding         │ 28.7 K │ train │     0 │
│ 1 │ embedding_dropout │ Dropout           │      0 │ train │     0 │
│ 2 │ lstm              │ LSTM              │  231 K │ train │     0 │
│ 3 │ classifier        │ Sequential        │  8.3 K │ train │     0 │
│ 4 │ loss_fn           │ BCEWithLogitsLoss │      0 │ train │     0 │
└───┴───────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 268 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 268 K                                                                                                
Total estimated model params size (MB): 1.074                                                                      
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Test metrics at threshold 0.5
accuracy: 0.9920490917616358
precision: 0.9990315757282295
recall: 0.9911005486309509
f1: 0.9950502589095339
roc_auc: 0.9978183088517871
pr_auc: 0.9995129555203688
false_positive: 38
false_negative: 352
attack_recall: 0.9911005486309509
attack_f1: 0.9950502589095339
Confusion matrix:
 [[ 9460    38]
 [  352 39201]]
Classification report:
               precision    recall  f1-score   support

      Normal       0.96      1.00      0.98      9498
      Attack       1.00      0.99      1.00     39553

    accuracy                           0.99     49051
   macro avg       0.98      0.99      0.99     49051
weighted avg       0.99      0.99      0.99     49051


Test metrics at validation-tuned threshold 0.38
accuracy: 0.9920694787058367
precision: 0.9989044585987261
recall: 0.9912522438247415
f1: 0.9950636397091481
roc_auc: 0.9978183088517871
pr_auc: 0.9995129555203688
false_positive: 43
false_negative: 346
attack_recall: 0.9912522438247415
attack_f1: 0.99

In [7]:
from google.colab import files
import shutil

shutil.make_archive('lstm_baseline_outputs', 'zip', '/content/drive/MyDrive/LSTM-Only/outputs/lstm_baseline')
files.download('lstm_baseline_outputs.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>